In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

In [2]:
'''df = pd.read_parquet(
    '../data/res/int.parquet',
    columns=['rating', 'user_id', 'parent_asin', 'date'])

df = df[df['date'] >= np.datetime64('2021-07-14')].copy()
df.to_parquet('new_df.parquet')'''

"df = pd.read_parquet(\n    '../data/res/int.parquet',\n    columns=['rating', 'user_id', 'parent_asin', 'date'])\n\ndf = df[df['date'] >= np.datetime64('2021-07-14')].copy()\ndf.to_parquet('new_df.parquet')"

решил сократит объем данных до последних 2 лет для экономии ОЗУ, как было выявлено на бейзлайне чем больше окно тем хуже рекомендации. но тк не факт что это работает для всех алгоритмов, поэтому окно остается большим. и потом можно проверить меняется ли качетство от времени

В качестве второго фильтра была выбрана Коллаборативная фильтрация чтобы оценивать поведение похожих пользователей. для этого выбрал алгоритм ALS тк у нас очень много пропусков в таблице и очень много юзеров и товаров поэтому матрица будет иметь много нулей. поэтому для экономии памяти будет использоваться разряженная матрица.  а ALS с разреженной матрцей работает стабильнее и экономнее

In [3]:
df = pd.read_parquet('new_df.parquet')

df = df.sort_values('date').reset_index(drop=True)

train = df[
    (df['date'] >= np.datetime64('2021-07-14')) &
    (df['date'] < np.datetime64('2023-07-14'))
].copy()

val = df[
    (df['date'] >= np.datetime64('2023-07-14')) &
    (df['date'] < np.datetime64('2023-08-14'))
].copy()

In [4]:
print(
    'Train period:',
    train['date'].min(),
    '->',
    train['date'].max()
)

print('Train rows:', len(train))
print('Train users:', train['user_id'].nunique())
print('Train items:', train['parent_asin'].nunique())

train['rating'].value_counts(normalize=True).sort_index()

Train period: 2021-07-14 00:00:07.199000 -> 2023-07-13 23:59:23.694000
Train rows: 10512499
Train users: 6282718
Train items: 1431877


rating
1.0    0.132925
2.0    0.052332
3.0    0.065216
4.0    0.105159
5.0    0.644369
Name: proportion, dtype: float64

In [5]:
val_pos = val[
    val['rating'] >= 4
].copy()

train_users = set(train['user_id'])

warm_val = val_pos[
    val_pos['user_id'].isin(train_users)
].copy()

print('Positive validation users:', val_pos['user_id'].nunique())
print('Warm validation users:', warm_val['user_id'].nunique())

Positive validation users: 47497
Warm validation users: 8790


ALS может строить персональные рекомендации только тем пользователям которых видела в обучении

In [6]:
train_seen = train[
    ['user_id', 'parent_asin']
].drop_duplicates()

warm_target = warm_val.merge(
    train_seen.assign(seen=1),
    on=['user_id', 'parent_asin'],
    how='left')

warm_target = warm_target[warm_target['seen'].isna()]
warm_user_targets = warm_target.groupby('user_id')['parent_asin'].agg(set)

print('Users for evaluation:', len(warm_user_targets))

Users for evaluation: 8785


In [7]:
def hit_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    return int(
        len(set(recom) & true_ans) > 0
    )

def recall_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans) 

    if len(true_ans) ==0:
        return 0
    
    hits = len(set(recom) & true_ans)

    return hits / len(true_ans)

In [8]:
als_data = (train[['user_id', 'parent_asin', 'rating', 'date']].sort_values('date')
            .drop_duplicates(['user_id', 'parent_asin'],keep='last').copy())

print('Unique user-item pairs:', len(als_data))

als_data['rating'].value_counts().sort_index()

Unique user-item pairs: 10506941


rating
1.0    1396729
2.0     549946
3.0     685270
4.0    1104869
5.0    6770127
Name: count, dtype: int64

In [9]:
als_data = als_data[als_data['rating'] >= 4].copy()

als_data['weight'] = 1.0

print('Positive interactions:', len(als_data))
print('Users:', als_data['user_id'].nunique())
print('Items:', als_data['parent_asin'].nunique())

Positive interactions: 7874996
Users: 4744979
Items: 1229644


на пересечении пары будет стоять 1 если отзыв был не менее 4 и 0 если меньшее либо если нет инфы. позже буду менять веса

In [10]:
user_codes, user_ids = pd.factorize(als_data['user_id'])
item_codes, item_ids = pd.factorize(als_data['parent_asin'])

In [11]:
user_item = csr_matrix((als_data['weight'].values,
                        (user_codes, item_codes)),
                        shape=( len(user_ids), len(item_ids)))

print('User-item matrix:', user_item.shape)
print('Interactions:', user_item.nnz)

User-item matrix: (4744979, 1229644)
Interactions: 7874996


In [12]:
model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    alpha=20,
    iterations=30,
    random_state=777
)

model.fit(
    user_item,
    show_progress=True
)

/home/user/mle/venv/lib/python3.14/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

In [13]:
user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

users = [
    user_id
    for user_id in warm_user_targets.index
    if user_id in user_to_idx
]

user_idx = np.array([
    user_to_idx[user_id]
    for user_id in users
])

print('Users with ALS predictions:', len(users))

Users with ALS predictions: 7586


In [14]:
rec_idx, _ = model.recommend(
    userid=user_idx,
    user_items=user_item[user_idx],
    N=1000,
    filter_already_liked_items=True
)

In [15]:
recommendations = {
    user_id: item_ids[item_idx].tolist()
    for user_id, item_idx in zip(users, rec_idx)
}

In [16]:
k_values = [10, 50, 100, 300, 1000]

rows = []

for k in k_values:

    hit_scores = []
    recall_scores = []

    for user_id, recs in recommendations.items():

        true_items = warm_user_targets[user_id]

        hit_scores.append(hit_at_k(recs, true_items, k))
        recall_scores.append(recall_at_k(recs, true_items, k))

    rows.append({
        'k': k,
        'hit_at_k': np.mean(hit_scores),
        'recall_at_k': np.mean(recall_scores)
    })

quality = pd.DataFrame(rows)

quality

,k,hit_at_k,recall_at_k
0,10,0.011468,0.008836
1,50,0.026233,0.019966
2,100,0.037437,0.028308
3,300,0.062352,0.049875
4,1000,0.121935,0.100592


теперь попробую разные способы построения матрицы взаимодействий

In [17]:
def train_als(
        train,
        days=np.nan,
        mode=1,
        factors=128,
        regularization=0.05,
        alpha=20,
        iterations=30):

    data = train

    if not pd.isna(days):
        end_date = train['date'].max()

        data = train[train['date'] >= end_date - pd.Timedelta(days=days)]

    data = (
        data[['user_id', 'parent_asin', 'rating', 'date']].sort_values('date')
        .drop_duplicates(['user_id', 'parent_asin'],keep='last').copy()
            )

    if mode == 1:

        data = data[data['rating'] >= 4].copy()

        data['weight'] = 1.0

    elif mode == 2:

        data['weight'] = np.select([data['rating'] <= 2, data['rating'] >= 4], 
                                   [-1.0, 1.0], 
                                   default=0.0)

        data = data[data['weight'] != 0]

    elif mode == 3:

        data['weight'] = data['rating'].map({
            1: -2.0,
            2: -1.0,
            3: 0.0,
            4: 1.0,
            5: 2.0
        })

        data = data[data['weight'] != 0]

    user_codes, user_ids = pd.factorize(data['user_id'])
    item_codes, item_ids = pd.factorize(data['parent_asin'])

    user_item = csr_matrix(
        (data['weight'].values, 
        (user_codes, item_codes)),
        shape=(len(user_ids),len(item_ids))
        )

    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        alpha=alpha,
        iterations=iterations,
        random_state=777
    )

    model.fit(user_item, show_progress=False)

    return model, user_item, user_ids, item_ids

мод 1 этот тот мод который был до этого. где вес 1 присваивался тем отзывам с оценками не ниже 4

мод 2 присваивает вес -1 для оценок ниже 3 а +1 выше 3. при этом оценка 3 отбрасывает. по сути присваивается 0. тк мы не можем точно сказать что товар понравился или нет. 

мод 3 присваивает веса в соответсвии со словарем {1: -2.0 ,
            2: -1.0 ,
            3: 0.0 ,
            4: 1.0 ,
            5: 2.0}

In [18]:
def predict_als(
        model,
        user_item,
        user_ids,
        item_ids,
        users,
        k):

    user_to_idx = {
        user_id: idx
        for idx, user_id in enumerate(user_ids)
    }

    users = [
        user_id
        for user_id in users
        if user_id in user_to_idx
    ]

    user_idx = np.array([
        user_to_idx[user_id]
        for user_id in users
    ])

    rec_idx, _ = model.recommend(
        userid=user_idx,
        user_items=user_item[user_idx],
        N=k,
        filter_already_liked_items=True
    )

    recommendations = {
        user_id: item_ids[item_idx].tolist()
        for user_id, item_idx in zip(users, rec_idx)
    }

    return recommendations

In [19]:
def evaluate_als(
        recommendations,
        user_targets,
        k):

    hit_scores = []
    recall_scores = []

    for user_id, recs in recommendations.items():
        true_items = user_targets[user_id]

        hit_scores.append(hit_at_k(recs, true_items, k))
        recall_scores.append(recall_at_k(recs, true_items, k))

    return {
        'hit_at_k': np.mean(hit_scores),
        'recall_at_k': np.mean(recall_scores)
    }

In [20]:
model, user_item, user_ids, item_ids = train_als(
    train=train,
    days=90,
    mode=1
)

In [21]:
recommendations = predict_als(
    model=model,
    user_item=user_item,
    user_ids=user_ids,
    item_ids=item_ids,
    users=warm_user_targets.index,
    k=1000
)

In [22]:
evaluate_als(
    recommendations=recommendations,
    user_targets=warm_user_targets,
    k=1000
)

{'hit_at_k': np.float64(0.16113744075829384),
 'recall_at_k': np.float64(0.12801700599779922)}

In [23]:
days_values = [360, 180, 90, 30]

k_values = [10, 50, 100, 300, 1000]

modes = [1, 2, 3]

In [24]:
results = []

for mode in modes:

    for days in days_values:

        model, user_item, user_ids, item_ids = train_als(train=train, days=days, mode=mode)

        recommendations = predict_als(
            model=model,
            user_item=user_item,
            user_ids=user_ids,
            item_ids=item_ids,
            users=warm_user_targets.index,
            k=max(k_values))

        for k in k_values:

            metrics = evaluate_als(recommendations=recommendations, user_targets=warm_user_targets, k=k)

            results.append({
                'mode': mode,
                'k': k,
                'days': days,
                'hit_at_k': metrics['hit_at_k'],
                'recall_at_k': metrics['recall_at_k']
            })

In [26]:
als_results = pd.DataFrame(results)

als_results[als_results['mode'] == 2]

,mode,k,days,hit_at_k,recall_at_k
20,2,10,360,0.017933,0.014885
21,2,50,360,0.037177,0.030197
22,2,100,360,0.050736,0.041490
23,2,300,360,0.081353,0.065753
24,2,1000,360,0.135005,0.111268
25,2,10,180,0.022707,0.017653
26,2,50,180,0.047036,0.037133
27,2,100,180,0.059650,0.047395
28,2,300,180,0.092810,0.074678
29,2,1000,180,0.148315,0.120331


In [27]:
als_results[als_results['mode'] == 3]

,mode,k,days,hit_at_k,recall_at_k
40,3,10,360,0.020411,0.016429
41,3,50,360,0.041551,0.033376
42,3,100,360,0.055547,0.045100
43,3,300,360,0.083248,0.067675
44,3,1000,360,0.138067,0.112869
45,3,10,180,0.024509,0.019273
46,3,50,180,0.047937,0.038677
47,3,100,180,0.065057,0.052144
48,3,300,180,0.092629,0.074316
49,3,1000,180,0.147955,0.121300


Как видно из представленных таблиц, лучшие результаты достигаются при использовании окна взаимодействий за последние 90 дней. Также качество возрастает при увеличении числа генерируемых рекомендаций и достигает максимума при k = 1000. Это ожидаемо, поскольку при увеличении размера списка кандидатов повышается вероятность того, что релевантный товар попадёт в рекомендации.


In [28]:
als_results[als_results['k'] == 1000]

,mode,k,days,hit_at_k,recall_at_k
4,1,1000,360,0.135428,0.111336
9,1,1000,180,0.154515,0.124142
14,1,1000,90,0.161137,0.128017
19,1,1000,30,0.116833,0.085207
24,2,1000,360,0.135005,0.111268
29,2,1000,180,0.148315,0.120331
34,2,1000,90,0.156150,0.122206
39,2,1000,30,0.115568,0.088622
44,3,1000,360,0.138067,0.112869
49,3,1000,180,0.147955,0.121300


Мод 1 (проставление веса 1 только для оценок не ниже 4) оказался самым лучшим. его качество при окне в 90 дней 0.16 для hit_at_k и 0.128 для recall_at_k. Поэтому в итоговой версии будем использовать для алс параметвы мод=1 k=1000 и days=90